# full pipeline
Setup and imports

Paths and task discovery from curated/v3.0/dataset

Flexible sequence and label extraction per task

Dataframe loader and fold builder

Tokenizer, model, metrics

Per-phase training function

Main curriculum training loop

# 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q transformers datasets accelerate rich pyarrow scikit-learn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 14.5 MB/s eta 0:00:00


In [ ]:
# === Step 0: CUDA allocator configuration and imports ===
import os

# Help PyTorch reduce fragmentation on long-running jobs
# See: https://pytorch.org/docs/stable/notes/cuda.html#environment-variables
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import math
from pathlib import Path

import torch
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import concatenate_datasets

from rich import print as rprint


In [ ]:
# Environment fix for DNABERT-2 Triton flash attention on Colab

import importlib
import subprocess, sys

spec = importlib.util.find_spec("triton")
if spec is not None:
    print("Uninstalling triton to avoid DNABERT-2 flash attention conflict...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "triton"], check=False)
else:
    print("triton package not found, no uninstall needed.")


Uninstalling triton to avoid DNABERT-2 flash attention conflict...


In [ ]:
import os, random, math, json, re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score
)

import torch
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from rich import print as rprint

# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
BASE = Path("/content/drive/MyDrive/MitoGPT")
CURATED_ROOT = BASE / "curated"
DATA_VERSION = "v3.0"
DATA_ROOT = CURATED_ROOT / DATA_VERSION / "dataset"

print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())

# Tasks present in the v3.0 layout
TASKS = [
    "human_gene_boundary",
    "human_gene_type",
    "human_regulatory",
    "human_variant_cls",
    "multispecies_gene_type",
]

# Curriculum phases
PHASE_TASKS = {
    1: ["human_gene_boundary", "human_gene_type"],   # basic human gene tasks
    2: ["human_regulatory", "human_variant_cls"],    # regulatory and variant tasks
    3: ["multispecies_gene_type"],                   # cross species generalization
}

# Model configuration
MODELS = ["zhihan1996/DNABERT-2-117M"]

DEBUG_SMALL = False  # set True for a quick smoke test

if DEBUG_SMALL:
    N_FOLDS = 2
    EPOCHS_PHASE = {1: 1, 2: 1, 3: 1}
    MAX_LEN = 512
    BATCH = 4
    GRAD_ACC = 2
else:
    N_FOLDS = 5
    EPOCHS_PHASE = {1: 2, 2: 2, 3: 1}
    MAX_LEN = 4096
    BATCH = 16
    GRAD_ACC = 4      # effective batch 64

LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01

print("DEBUG_SMALL:", DEBUG_SMALL,
      "| N_FOLDS:", N_FOLDS,
      "| MAX_LEN:", MAX_LEN,
      "| BATCH:", BATCH,
      "| GRAD_ACC:", GRAD_ACC)


DATA_ROOT: /content/drive/MyDrive/MitoGPT/curated/v3.0/dataset
DATA_ROOT exists: True
DEBUG_SMALL: False | N_FOLDS: 5 | MAX_LEN: 4096 | BATCH: 16 | GRAD_ACC: 4


# 2. Discover tasks and parquet files

In [ ]:
def build_task_file_map(dataset_root: Path) -> Dict[str, List[Path]]:
    """
    Scan dataset_root for task=... folders and gather all parquet files recursively.
    """
    task_file_map: Dict[str, List[Path]] = {}

    for task_dir in sorted(dataset_root.glob("task=*")):
        if not task_dir.is_dir():
            continue
        task_name = task_dir.name.split("=", 1)[1]
        files = sorted(task_dir.rglob("*.parquet"))
        if not files:
            print(f"[WARN] task={task_name} has no parquet files under {task_dir}")
            continue
        task_file_map[task_name] = files
        print(f"[OK] task={task_name:24s} n_files={len(files):3d} example={files[0].name}")

    if not task_file_map:
        raise RuntimeError(f"No task=... folders found under {dataset_root}")

    return task_file_map

TASK_FILE_MAP = build_task_file_map(DATA_ROOT)
print("Tasks discovered on disk:", sorted(TASK_FILE_MAP))

[OK] task=human_gene_boundary      n_files=  3 example=3e4a0b887f444d9182f7fd78702b8f5e-0.parquet
[OK] task=human_gene_type          n_files=  3 example=3e4a0b887f444d9182f7fd78702b8f5e-0.parquet
[OK] task=human_regulatory         n_files=  3 example=acc599218cd74bcc95f91a59cd3d93d5-0.parquet
[OK] task=human_variant_cls        n_files=  6 example=66c18a0d05844ce5b6cbd010b96e0db8-0.parquet
[OK] task=multispecies_gene_type   n_files= 80 example=7f2b42f4113b4b1c9d6c03899f30d70b-0.parquet
Tasks discovered on disk: ['human_gene_boundary', 'human_gene_type', 'human_regulatory', 'human_variant_cls', 'multispecies_gene_type']


In [ ]:
def list_parquet_files(task: str) -> List[Path]:
    """
    Return the list of parquet files for a given logical task name.
    """
    if task not in TASK_FILE_MAP:
        available = sorted(TASK_FILE_MAP)
        raise FileNotFoundError(
            f"Task {task!r} not found under {DATA_ROOT}. "
            f"Available tasks: {available}"
        )
    return TASK_FILE_MAP[task]

# Optional sanity check
def quick_row_count(task: str, max_files: int = 3):
    files = list_parquet_files(task)
    sample_files = files[:max_files]
    total_rows = 0
    for f in sample_files:
        table = pq.read_table(f)
        total_rows += table.num_rows
    print(f"task={task:24s} sample_files={len(sample_files)} rows_in_sample={total_rows}")

for t in TASKS:
    quick_row_count(t)

task=human_gene_boundary      sample_files=3 rows_in_sample=111
task=human_gene_type          sample_files=3 rows_in_sample=111
task=human_regulatory         sample_files=3 rows_in_sample=36
task=human_variant_cls        sample_files=3 rows_in_sample=3207
task=multispecies_gene_type   sample_files=3 rows_in_sample=421


# 3. Helpers - including automatic sequence column detection

In [ ]:
def safe_isnan(x):
    try:
        return math.isnan(x)
    except Exception:
        return False


In [ ]:
DNA_CHARS = set("ACGTNacgtn")

def is_dna_like(s: str, min_len: int = 30, min_fraction: float = 0.8) -> bool:
    """
    Test if a string looks like a DNA sequence.
    """
    if s is None:
        return False
    if not isinstance(s, str):
        s = str(s)
    s = s.strip()
    if len(s) < min_len:
        return False
    total = len(s)
    dna_count = sum(1 for ch in s if ch in DNA_CHARS)
    if total == 0:
        return False
    frac = dna_count / total
    return frac >= min_fraction


def detect_sequence_column(df: pd.DataFrame) -> str:
    """
    Automatically detect which column holds the DNA sequence.

    Strategy:
    - candidate columns are object or string dtype
    - exclude obvious metadata columns
    - sample values and pick the first that looks like DNA sequence
    """
    exclude_exact = {
        "label",
        "task",
        "group",
        "label_str",
        "species",
        "gene_id",
        "region_id",
    }
    exclude_prefixes = [
        "functional_annotations",
        "gene_annotations",
        "variant_annotations",
        "meta",
    ]

    candidates = []
    for col in df.columns:
        if col in exclude_exact:
            continue
        if any(col.startswith(pref) for pref in exclude_prefixes):
            continue
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col].dtype):
            candidates.append(col)

    # If known good names exist, prefer them first
    preferred_names = ["sequence", "seq", "window_seq", "nt_sequence", "dna"]
    for name in preferred_names:
        if name in candidates:
            # quick dna check
            sample_vals = df[name].dropna().astype(str).head(20).tolist()
            if sample_vals and any(is_dna_like(v) for v in sample_vals):
                print(f"[SEQ] using preferred column {name}")
                return name

    # Otherwise test candidates by content
    for col in candidates:
        sample_vals = df[col].dropna().astype(str).head(20).tolist()
        if not sample_vals:
            continue
        hits = sum(1 for v in sample_vals if is_dna_like(v))
        if hits >= max(3, int(0.5 * len(sample_vals))):
            print(f"[SEQ] detected sequence column {col}")
            return col

    # If nothing matched, print some debug info and fail
    print("Could not detect DNA sequence column.")
    print("Columns:", list(df.columns))
    print("Head row:")
    print(df.head(1).T)
    raise RuntimeError("No sequence-like column detected in dataframe")


Label extractors from annotations:

In [ ]:
# Gene type categories
GENE_TYPE_KEYS = ("gene_type", "feature_type", "biotype", "type", "kind", "category")

def extract_gene_type_from_mlaa(mlaa: dict) -> Optional[str]:
    """
    For gene_type tasks, extract a canonical gene type string from mlaa_json.

    Expected structure (conceptual):

      {
        "sequence": "...",
        "coords": {...},
        "gene_annotations": [ {...}, {...}, ... ],
        "functional_annotations": {...},
        "variant_annotations": [...]
      }
    """
    if mlaa is None or safe_isnan(mlaa):
        return None

    ga = mlaa.get("gene_annotations")
    types = []

    if isinstance(ga, dict):
        for k in GENE_TYPE_KEYS:
            if k in ga and ga[k] not in (None, "", "NA", "NaN"):
                types.append(str(ga[k]))
        for v in ga.values():
            if isinstance(v, str) and v.strip():
                types.append(v.strip())

    elif isinstance(ga, (list, tuple)):
        for item in ga:
            if isinstance(item, dict):
                for k in GENE_TYPE_KEYS:
                    if k in item and item[k] not in (None, "", "NA", "NaN"):
                        types.append(str(item[k]))
            elif isinstance(item, str) and item.strip():
                types.append(item.strip())
            elif item not in (None, "", "NA", "NaN"):
                types.append(str(item))

    elif isinstance(ga, str):
        if ga.strip():
            types.append(ga.strip())

    if not types:
        return None

    types = [t.strip().lower() for t in types if t and not safe_isnan(t)]
    if not types:
        return None

    if any("protein_coding" in t or "protein-coding" in t for t in types):
        return "protein_coding"
    if any("trna" in t for t in types):
        return "tRNA"
    if any("rrna" in t for t in types):
        return "rRNA"
    if any("lncrna" in t or "long_noncoding" in t or "lnc_rna" in t for t in types):
        return "lncRNA"

    return types[0]


# 4. Data loading per task and fold building

In [ ]:
# Variant classification labels

VAR_CLS_KEYS = [
    "pathogenicity_label",
    "clinical_significance",
    "pathogenicity",
    "mitomap_class",
]

def extract_variant_cls_from_mlaa(mlaa: dict) -> Optional[str]:
    """
    Extract a pathogenicity like label from mlaa_json for human_variant_cls.
    """
    if mlaa is None or safe_isnan(mlaa):
        return None

    va = mlaa.get("variant_annotations")
    if va is None:
        return None

    labels = []

    if isinstance(va, dict):
        for k in VAR_CLS_KEYS:
            if k in va and va[k] is not None:
                labels.append(str(va[k]))
        if "clinical_significance" in va and isinstance(va["clinical_significance"], (list, tuple)):
            for x in va["clinical_significance"]:
                if isinstance(x, str) and x.strip():
                    labels.append(x.strip())

    elif isinstance(va, (list, tuple)):
        for item in va:
            if isinstance(item, dict):
                for k in VAR_CLS_KEYS:
                    if k in item and item[k] is not None:
                        labels.append(str(item[k]))
                if "clinical_significance" in item and isinstance(item["clinical_significance"], (list, tuple)):
                    for x in item["clinical_significance"]:
                        if isinstance(x, str) and x.strip():
                            labels.append(x.strip())
            elif isinstance(item, str) and item.strip():
                labels.append(item.strip())
            elif item not in (None, "", "NA", "NaN"):
                labels.append(str(item))

    elif isinstance(va, str):
        labels.append(va.strip())
    else:
        labels.append(str(va))

    labels = [l.strip().lower() for l in labels if l and not safe_isnan(l)]
    if not labels:
        return None

    for l in labels:
        if "pathogenic" in l and "likely" not in l and "non" not in l:
            return "pathogenic"
    for l in labels:
        if "likely pathogenic" in l or "likely_pathogenic" in l:
            return "likely_pathogenic"
    for l in labels:
        if "benign" in l and "likely" not in l and "non" not in l:
            return "benign"
    for l in labels:
        if "likely benign" in l or "likely_benign" in l:
            return "likely_benign"
    for l in labels:
        if "polymorphism" in l:
            return "polymorphism"
    for l in labels:
        if "vus" in l or "uncertain" in l:
            return "uncertain"

    return labels[0]


In [ ]:
def extract_regulatory_label_from_mlaa(mlaa: dict) -> Optional[int]:
    """
    Binary label for human_regulatory from mlaa_json.

    1 if functional_annotations.regulatory_elements has any entries.
    """
    if mlaa is None or safe_isnan(mlaa):
        return None

    fa = mlaa.get("functional_annotations") or {}
    reg = fa.get("regulatory_elements")
    if reg is None:
        return 0

    if isinstance(reg, (list, tuple, dict)):
        return int(len(reg) > 0)
    if isinstance(reg, str):
        return int(bool(reg.strip()))
    return int(bool(reg))


In [ ]:
def extract_boundary_label_from_mlaa(mlaa: dict) -> Optional[int]:
    """
    Binary label for human_gene_boundary from mlaa_json.

    1 if functional_annotations.processing_sites has any entries.
    """
    if mlaa is None or safe_isnan(mlaa):
        return None

    fa = mlaa.get("functional_annotations") or {}
    ps = fa.get("processing_sites")
    if ps is None:
        return 0

    if isinstance(ps, (list, tuple, dict)):
        return int(len(ps) > 0)
    if isinstance(ps, str):
        return int(bool(ps.strip()))
    return int(bool(ps))


In [ ]:
def load_task_df(task: str, limit_rows: Optional[int] = None) -> pd.DataFrame:
    """
    Load parquet files for a task and parse mlaa_json into:
      - sequence: str
      - label: int
      - task: str
      - group: str (species or gene etc)

    Assumes each parquet has columns:
      mlaa_json, task, tier, source, species, date
    where mlaa_json is a JSON string with keys such as:
      sequence, coords, gene_annotations,
      functional_annotations, variant_annotations.
    """
    files = list_parquet_files(task)
    dfs = []
    total_rows = 0

    for f in files:
        table = pq.read_table(f)
        df_part = table.to_pandas()
        dfs.append(df_part)
        total_rows += len(df_part)
        if limit_rows is not None and total_rows >= limit_rows:
            break

    if not dfs:
        raise RuntimeError(f"No rows loaded for task {task}")

    df = pd.concat(dfs, ignore_index=True)

    if DEBUG_SMALL and len(df) > 2000:
        df = df.sample(2000, random_state=SEED).reset_index(drop=True)

    if "mlaa_json" not in df.columns:
        raise KeyError(f"Column 'mlaa_json' missing in task {task} dataframe")

    # Parse mlaa_json into dicts
    def parse_mlaa(x):
        if isinstance(x, dict):
            return x
        if isinstance(x, str):
            return json.loads(x)
        raise ValueError(f"Unexpected mlaa_json type: {type(x)}")

    df["mlaa"] = df["mlaa_json"].apply(parse_mlaa)

    # Extract sequence from mlaa
    df["sequence"] = df["mlaa"].apply(lambda d: d.get("sequence", ""))

    # Quick sanity print for the first task that is loaded
    print(f"[SEQ] task={task} example sequence (first row, first 60 nt):")
    print(df["sequence"].iloc[0][:60])

    # Build labels per task
    if task in ("human_gene_type", "multispecies_gene_type"):
        labels_str = df["mlaa"].apply(extract_gene_type_from_mlaa)
        df["label_str"] = labels_str
        df = df[df["label_str"].notna()].copy()
        uniq = sorted(df["label_str"].unique().tolist())
        mapping = {lab: i for i, lab in enumerate(uniq)}
        print(f"[LBL] gene type mapping for {task}: {mapping}")
        df["label"] = df["label_str"].map(mapping).astype(int)

    elif task == "human_variant_cls":
        labels_str = df["mlaa"].apply(extract_variant_cls_from_mlaa)
        df["label_str"] = labels_str
        df = df[df["label_str"].notna()].copy()
        uniq = sorted(df["label_str"].unique().tolist())
        mapping = {lab: i for i, lab in enumerate(uniq)}
        print(f"[LBL] variant mapping for {task}: {mapping}")
        df["label"] = df["label_str"].map(mapping).astype(int)

    elif task == "human_regulatory":
        labels = df["mlaa"].apply(extract_regulatory_label_from_mlaa)
        df["label"] = labels
        df = df[df["label"].notna()].copy()
        df["label"] = df["label"].astype(int)

    elif task == "human_gene_boundary":
        labels = df["mlaa"].apply(extract_boundary_label_from_mlaa)
        df["label"] = labels
        df = df[df["label"].notna()].copy()
        df["label"] = df["label"].astype(int)

    else:
        raise ValueError(f"Unhandled task {task}")

    if len(df) == 0:
        raise RuntimeError(f"Task {task} has 0 rows after label extraction")

    df["task"] = task

    # Group for fold splitting
    if "species" in df.columns and df["species"].notna().any():
        df["group"] = df["species"].astype(str)
    else:
        df["group"] = "global"

    return df


In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold

def make_folds(df: pd.DataFrame, n_splits: int = N_FOLDS) -> List[DatasetDict]:
    """
    Build train, validation, test splits.

    Logic:
    - If group column exists and has enough groups, use GroupKFold.
    - Else, if labels are multi class and each class has enough samples, use StratifiedKFold.
    - Else, fall back to simple KFold.
    """
    y = df["label"].values
    unique_labels, counts = np.unique(y, return_counts=True)
    n_classes = len(unique_labels)

    has_group = "group" in df.columns
    n_groups = df["group"].nunique() if has_group else 0

    # Outer split selection
    if has_group and n_groups > 1 and n_groups >= n_splits:
        print(f"[FOLDS] Using GroupKFold: n_groups={n_groups}, n_splits={n_splits}")
        splitter = GroupKFold(n_splits=n_splits)
        split_iter = splitter.split(df, y, df["group"].values)
    else:
        if n_classes >= 2 and counts.min() >= n_splits:
            print(f"[FOLDS] Using StratifiedKFold: n_classes={n_classes}, n_splits={n_splits}")
            splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
            split_iter = splitter.split(df, y)
        else:
            # Single label tasks or very imbalanced tasks
            k = min(n_splits, len(df))
            print(f"[FOLDS] Using plain KFold: n_samples={len(df)}, n_splits={k}, n_classes={n_classes}")
            splitter = KFold(n_splits=k, shuffle=True, random_state=SEED)
            split_iter = splitter.split(df)

    folds = []
    for fold_idx, (trainval_idx, test_idx) in enumerate(split_iter):
        trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
        test_df     = df.iloc[test_idx].reset_index(drop=True)

        y_tv = trainval_df["label"].values
        unique_tv, counts_tv = np.unique(y_tv, return_counts=True)
        n_classes_tv = len(unique_tv)

        # Inner split for train/validation
        if n_classes_tv >= 2 and counts_tv.min() >= 5:
            inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + fold_idx)
            print(f"[INNER] Fold {fold_idx}: StratifiedKFold inner split, n_classes={n_classes_tv}")
            inner_train_idx, inner_val_idx = next(inner.split(trainval_df, y_tv))
        else:
            k_inner = min(5, len(trainval_df))
            inner = KFold(n_splits=k_inner, shuffle=True, random_state=SEED + fold_idx)
            print(f"[INNER] Fold {fold_idx}: KFold inner split, n_splits={k_inner}, n_classes={n_classes_tv}")
            inner_train_idx, inner_val_idx = next(inner.split(trainval_df))

        train_df = trainval_df.iloc[inner_train_idx].reset_index(drop=True)
        val_df   = trainval_df.iloc[inner_val_idx].reset_index(drop=True)

        ds_train = Dataset.from_pandas(train_df, preserve_index=False)
        ds_val   = Dataset.from_pandas(val_df, preserve_index=False)
        ds_test  = Dataset.from_pandas(test_df, preserve_index=False)

        folds.append(DatasetDict(train=ds_train, validation=ds_val, test=ds_test))

    return folds

In [ ]:
def build_task_folds() -> Tuple[Dict[str, List[DatasetDict]], Dict[str, int], List[str]]:
    """
    For each task, load dataframe, build folds and record number of labels.

    Single label tasks are included. Metrics for those tasks are trivial, but
    they still contribute to shared representation learning.
    """
    task_folds: Dict[str, List[DatasetDict]] = {}
    task_num_labels: Dict[str, int] = {}
    active_tasks: List[str] = []

    for t in TASKS:
        try:
            df = load_task_df(t)
            nlab = int(df["label"].nunique())
            print(f"[TASK] {t}: n={len(df)}, num_labels={nlab}")

            if nlab < 2:
                rprint(
                    f"[yellow]Task {t} has a single label. "
                    f"It will still be included in training but "
                    f"metrics will be trivial for this task.[/yellow]"
                )

            folds = make_folds(df, n_splits=N_FOLDS)
            task_folds[t] = folds
            task_num_labels[t] = nlab
            active_tasks.append(t)

        except Exception as e:
            rprint(f"[red]Task {t} failed during fold build:[/red] {e}")

    if not active_tasks:
        raise RuntimeError(
            "No tasks with usable labels after fold-building. "
            "Check parquet paths, mlaa_json parsing, and label rules."
        )

    return task_folds, task_num_labels, active_tasks


task_folds, task_num_labels, ACTIVE_TASKS = build_task_folds()
rprint(f"[bold green]Active tasks:[/bold green] {ACTIVE_TASKS}")

[SEQ] task=human_gene_boundary example sequence (first row, first 60 nt):
GTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTCACATCA
[TASK] human_gene_boundary: n=111, num_labels=1


Task human_gene_boundary has a single label. It will still be included in training but metrics will be trivial for 
this task.

[FOLDS] Using plain KFold: n_samples=111, n_splits=5, n_classes=1
[INNER] Fold 0: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 1: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 2: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 3: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 4: KFold inner split, n_splits=5, n_classes=1
[SEQ] task=human_gene_type example sequence (first row, first 60 nt):
GTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTCACATCA
[LBL] gene type mapping for human_gene_type: {'cds': 0, 'rRNA': 1, 'tRNA': 2}
[TASK] human_gene_type: n=111, num_labels=3
[FOLDS] Using StratifiedKFold: n_classes=3, n_splits=5
[INNER] Fold 0: StratifiedKFold inner split, n_classes=3
[INNER] Fold 1: KFold inner split, n_splits=5, n_classes=3
[INNER] Fold 2: StratifiedKFold inner split, n_classes=3
[INNER] Fold 3: StratifiedKFold inner split, n_classes=3
[INNER] Fold 4: StratifiedKFold inner split, n_classes=3
[SEQ] task=human_regulatory example sequence

Task human_regulatory has a single label. It will still be included in training but metrics will be trivial for 
this task.

[FOLDS] Using plain KFold: n_samples=36, n_splits=5, n_classes=1
[INNER] Fold 0: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 1: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 2: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 3: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 4: KFold inner split, n_splits=5, n_classes=1
[SEQ] task=human_variant_cls example sequence (first row, first 60 nt):
T
[LBL] variant mapping for human_variant_cls: {'[none]': 0}
[TASK] human_variant_cls: n=61371, num_labels=1


Task human_variant_cls has a single label. It will still be included in training but metrics will be trivial for 
this task.

[FOLDS] Using plain KFold: n_samples=61371, n_splits=5, n_classes=1
[INNER] Fold 0: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 1: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 2: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 3: KFold inner split, n_splits=5, n_classes=1
[INNER] Fold 4: KFold inner split, n_splits=5, n_classes=1
[SEQ] task=multispecies_gene_type example sequence (first row, first 60 nt):
TAACAAAGTAAGCTAATTTAAGCTAGTGGGTTCATACCCCAAAAATGAATTTTTCCTTTG
[LBL] gene type mapping for multispecies_gene_type: {'cds': 0, 'rRNA': 1, 'tRNA': 2}
[TASK] multispecies_gene_type: n=5000, num_labels=3
[FOLDS] Using GroupKFold: n_groups=80, n_splits=5
[INNER] Fold 0: StratifiedKFold inner split, n_classes=3
[INNER] Fold 1: StratifiedKFold inner split, n_classes=3
[INNER] Fold 2: StratifiedKFold inner split, n_classes=3
[INNER] Fold 3: StratifiedKFold inner split, n_classes=3
[INNER] Fold 4: StratifiedKFold inner split, n_classes=3


Active tasks: ['human_gene_boundary', 'human_gene_type', 'human_regulatory', 'human_variant_cls', 
'multispecies_gene_type']

# 5. Tokenizer, model, metrics

In [ ]:
# === STEP 5: Curriculum fine-tuning on DNABERT-2 ===

import os
from pathlib import Path
from typing import Dict, List, Tuple, Union, Any

import numpy as np
import torch
from datasets import concatenate_datasets
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EvalPrediction,
    set_seed,
)
from rich import print as rprint

# Slightly safer CUDA allocator configuration for Colab
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


'expandable_segments:True'

In [ ]:
def compute_metrics(
    eval_pred: Union[EvalPrediction, Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, float]:
    """
    Metric function compatible with Hugging Face Trainer.

    Handles:
      - EvalPrediction(predictions=..., label_ids=...)
      - Plain (predictions, labels) tuples
      - Predictions returned as a tuple or list, using the first element as logits
    """
    if isinstance(eval_pred, EvalPrediction):
        predictions = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        predictions, labels = eval_pred

    if isinstance(predictions, (tuple, list)):
        predictions = predictions[0]

    if isinstance(predictions, torch.Tensor):
        predictions = predictions.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    preds = predictions.argmax(axis=-1)

    # Optional mask support if -100 is ever used
    if isinstance(labels, np.ndarray) and labels.dtype == np.int64:
        mask = labels != -100
        if mask.any() and mask.shape == preds.shape:
            labels = labels[mask]
            preds = preds[mask]

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")

    return {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
    }


In [ ]:
def make_training_args(
    output_dir: Path,
    num_epochs: float,
    learning_rate: float = 5e-5,
    train_bs: int = 8,
    eval_bs: int = 8,
    grad_accum: int = 2,
    warmup_ratio: float = 0.1,
) -> TrainingArguments:
    """
    TrainingArguments for DNABERT-2 curriculum finetuning on Colab.

    Uses standard fields from current Transformers versions.
    """
    fp16_flag = torch.cuda.is_available()

    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=grad_accum,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=warmup_ratio,
        logging_steps=5,
        evaluation_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        fp16=fp16_flag,
        dataloader_num_workers=2,
        report_to=[],
        remove_unused_columns=False,
    )


In [ ]:
def get_tokenizer(model_id: str):
    """
    Load tokenizer for DNABERT-2 and ensure PAD/BOS/EOS tokens exist.

    This matches earlier steps in the notebook and is safe to call multiple times.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens(
            {
                "pad_token": "<pad>",
                "bos_token": "<s>",
                "eos_token": "</s>",
            }
        )

    return tokenizer


In [ ]:
def get_model(
    model_id: str,
    num_labels: int,
    tokenizer=None,
):
    """
    Load a classification model on top of DNABERT-2.

    The tokenizer argument is optional and ignored, so both of these calls are valid:
      get_model(model_id, num_labels)
      get_model(model_id, num_labels, tokenizer)
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        trust_remote_code=True,
    )
    return model


In [ ]:
def make_collator(tokenizer):
    """
    Padding collator for sequence classification datasets.
    """
    return DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )


In [ ]:
def encode_batch(batch, tokenizer, max_length: int):
    """
    Tokenization helper that maps 'sequence' to model inputs and attaches 'labels'.

    Assumes Step 4 has already produced datasets with 'sequence' and 'label' columns.
    """
    seqs = batch["sequence"]
    enc = tokenizer(
        seqs,
        truncation=True,
        max_length=max_length,
    )
    enc["labels"] = batch["label"]
    return enc


In [ ]:
def train_phase(
    model: torch.nn.Module,
    model_id: str,
    tokenizer,
    collator,
    fold_idx: int,
    phase_id: int,
    tasks_for_phase: List[str],
    results_rows: List[Dict[str, Any]],
    max_length: int = 512,
) -> torch.nn.Module:
    """
    Train one curriculum phase for one fold across a set of tasks.

    Returns the updated model so subsequent phases can continue from the same weights.
    """
    train_parts = []
    eval_parts = []

    for task_name in tasks_for_phase:
        folds_for_task = task_folds.get(task_name)

        if not folds_for_task:
            rprint(f"[yellow]Task {task_name} has no folds, skipping[/yellow]")
            continue

        if fold_idx >= len(folds_for_task):
            rprint(f"[yellow]Task {task_name} missing fold {fold_idx}, skipping[/yellow]")
            continue

        fold_data = folds_for_task[fold_idx]

        train_ds = fold_data.get("train")
        eval_ds = fold_data.get("eval")

        if train_ds is None or len(train_ds) == 0:
            rprint(f"[yellow]Task {task_name} fold {fold_idx} train empty, skipping[/yellow]")
            continue
        if eval_ds is None or len(eval_ds) == 0:
            rprint(f"[yellow]Task {task_name} fold {fold_idx} eval empty, skipping[/yellow]")
            continue

        train_parts.append(train_ds)
        eval_parts.append(eval_ds)

    if not train_parts or not eval_parts:
        rprint(f"[yellow]No datasets for phase {phase_id} fold {fold_idx}, skipping phase[/yellow]")
        return model

    train_dataset = concatenate_datasets(train_parts)
    eval_dataset = concatenate_datasets(eval_parts)

    rprint(
        f"[green]Tokenizing train fold {fold_idx} phase {phase_id}: "
        f"{len(train_dataset)} examples[/green]"
    )
    rprint(
        f"[green]Tokenizing eval fold {fold_idx} phase {phase_id}: "
        f"{len(eval_dataset)} examples[/green]"
    )

    # Tokenize and attach labels
    train_cols_to_remove = [
        c for c in train_dataset.column_names if c not in ["sequence", "label"]
    ]
    eval_cols_to_remove = [
        c for c in eval_dataset.column_names if c not in ["sequence", "label"]
    ]

    tokenized_train = train_dataset.map(
        lambda batch: encode_batch(batch, tokenizer, max_length),
        batched=True,
        remove_columns=train_cols_to_remove,
        desc=f"Tokenizing train fold {fold_idx} phase {phase_id}",
    )
    tokenized_eval = eval_dataset.map(
        lambda batch: encode_batch(batch, tokenizer, max_length),
        batched=True,
        remove_columns=eval_cols_to_remove,
        desc=f"Tokenizing eval fold {fold_idx} phase {phase_id}",
    )

    # Simple curriculum schedule
    if phase_id == 1:
        num_epochs = 2.0
    elif phase_id == 2:
        num_epochs = 1.5
    else:
        num_epochs = 1.0

    out_dir = (
        Path("mitogpt_runs_drive")
        / model_id.replace("/", "__")
        / f"fold{fold_idx}_phase{phase_id}"
    )
    out_dir.mkdir(parents=True, exist_ok=True)

    training_args = make_training_args(
        output_dir=out_dir,
        num_epochs=num_epochs,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    rprint(
        f"[cyan]Training {model_id} fold={fold_idx} phase={phase_id} "
        f"on tasks={tasks_for_phase}[/cyan]"
    )

    train_output = trainer.train()
    eval_output = trainer.evaluate()

    results_rows.append(
        {
            "model_id": model_id,
            "fold_idx": fold_idx,
            "phase_id": phase_id,
            "tasks": ",".join(tasks_for_phase),
            "n_train": len(tokenized_train),
            "n_eval": len(tokenized_eval),
            "train_loss": float(getattr(train_output, "training_loss", np.nan)),
            "eval_loss": float(eval_output.get("eval_loss", np.nan)),
            "eval_accuracy": float(eval_output.get("eval_accuracy", np.nan)),
            "eval_f1_macro": float(eval_output.get("eval_f1_macro", np.nan)),
        }
    )

    return model


In [ ]:
# Main curriculum loop

set_seed(SEED)

if not task_num_labels:
    raise RuntimeError("task_num_labels is empty. Step 4 must be run successfully first.")

num_labels_for_all = max(task_num_labels.values())
rprint(f"[bold green]Model head num_labels_for_all={num_labels_for_all}[/bold green]")

all_results: List[Dict[str, Any]] = []

for model_id in MODELS:
    for fold_idx in range(N_FOLDS):
        rprint(f"[bold yellow]=== {model_id} | fold {fold_idx} ===[/bold yellow]")

        tokenizer = get_tokenizer(model_id)
        collator = make_collator(tokenizer)

        # Fresh model per fold, phases continue from previous phase weights
        model = get_model(model_id, num_labels=num_labels_for_all)

        for phase_id in (1, 2, 3):
            phase_tasks = [t for t in PHASE_TASKS.get(phase_id, []) if t in ACTIVE_TASKS]
            if not phase_tasks:
                rprint(f"[yellow]No active tasks for phase {phase_id}, skipping[/yellow]")
                continue

            model = train_phase(
                model=model,
                model_id=model_id,
                tokenizer=tokenizer,
                collator=collator,
                fold_idx=fold_idx,
                phase_id=phase_id,
                tasks_for_phase=phase_tasks,
                results_rows=all_results,
                max_length=512,
            )

# Save curriculum summary
summary_df = pd.DataFrame(all_results)
summary_out = Path("mitogpt_runs_drive") / "mitogpt_curriculum_summary.csv"
summary_out.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_out, index=False)
rprint(f"[bold green]Saved curriculum summary to[/bold green] {summary_out}")


Model head num_labels_for_all=3

=== zhihan1996/DNABERT-2-117M | fold 0 ===

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Task human_gene_boundary fold 0 eval empty, skipping

Task human_gene_type fold 0 eval empty, skipping

...

Task human_variant_cls fold 4 eval empty, skipping

No datasets for phase 2 fold 4, skipping phase

Task multispecies_gene_type fold 4 eval empty, skipping

No datasets for phase 3 fold 4, skipping phase

Saved curriculum summary to mitogpt_runs_drive/mitogpt_curriculum_summary.csv

6. Training arguments and phase trainer

In [ ]:
# Step 6: Baseline data preparation and Trainer builder

import json
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

# Reproducibility
set_seed(42)

# Output root for baseline run
BASELINE_OUT_ROOT = Path("./mitogpt_runs_drive") / "baseline_gene_type"
BASELINE_OUT_ROOT.mkdir(parents=True, exist_ok=True)


def build_baseline_dataframe():
    """
    Build a unified gene_type dataframe for baseline finetuning.

    Uses human_gene_type and multispecies_gene_type tasks.
    Assumes load_task_df(task_name) is defined earlier in the notebook.
    """
    # Load both tasks; handle both DataFrame and (DataFrame, label2id) outputs
    res_human = load_task_df("human_gene_type")
    res_multi = load_task_df("multispecies_gene_type")

    df_human = res_human[0] if isinstance(res_human, tuple) else res_human
    df_multi = res_multi[0] if isinstance(res_multi, tuple) else res_multi

    if not isinstance(df_human, pd.DataFrame) or not isinstance(df_multi, pd.DataFrame):
        raise TypeError("load_task_df must return a pandas DataFrame or (DataFrame, label2id)")

    # Detect sequence column
    seq_col = None
    for cand in ["sequence", "seq", "dna", "nt"]:
        if cand in df_human.columns:
            seq_col = cand
            break
    if seq_col is None:
        raise KeyError(
            f"No sequence-like column found in human_gene_type dataframe. "
            f"Columns: {df_human.columns.tolist()}"
        )

    # Detect label column
    label_col = None
    for cand in ["gene_type", "label", "target"]:
        if cand in df_human.columns:
            label_col = cand
            break
    if label_col is None:
        raise KeyError(
            "No label column found in human_gene_type dataframe. "
            f"Columns: {df_human.columns.tolist()}"
        )

    # Ensure same columns exist in multispecies dataframe
    if seq_col not in df_multi.columns:
        raise KeyError(f"Sequence column '{seq_col}' missing in multispecies_gene_type dataframe")
    if label_col not in df_multi.columns:
        raise KeyError(f"Label column '{label_col}' missing in multispecies_gene_type dataframe")

    # Work on copies
    df_h = df_human[[seq_col, label_col]].copy()
    df_m = df_multi[[seq_col, label_col]].copy()

    # Unified label mapping across both datasets
    labels_h = list(pd.unique(df_h[label_col]))
    labels_m = list(pd.unique(df_m[label_col]))
    all_labels = sorted(set(labels_h) | set(labels_m), key=str)

    # If labels are already ints, keep them; if strings, create mapping
    if all(isinstance(x, (int, np.integer)) for x in all_labels):
        label2id = {int(x): int(x) for x in all_labels}
        df_h["label"] = df_h[label_col].astype(int)
        df_m["label"] = df_m[label_col].astype(int)
    else:
        label2id = {str(name): idx for idx, name in enumerate(all_labels)}
        df_h["label"] = df_h[label_col].astype(str).map(label2id).astype(int)
        df_m["label"] = df_m[label_col].astype(str).map(label2id).astype(int)

    print(f"[LBL] unified gene_type label2id: {label2id}")

    # Normalise sequence column name
    df_h["sequence"] = df_h[seq_col].astype(str)
    df_m["sequence"] = df_m[seq_col].astype(str)

    # Small provenance tag
    df_h["source_task"] = "human_gene_type"
    df_m["source_task"] = "multispecies_gene_type"

    df_all = pd.concat(
        [df_h[["sequence", "label", "source_task"]], df_m[["sequence", "label", "source_task"]]],
        ignore_index=True,
    )

    print(f"Baseline dataframe size: {len(df_all)} rows")
    return df_all, label2id


def build_hf_datasets(df_all, tokenizer, test_size=0.2, max_length=512):
    """
    Convert pandas dataframe into tokenized Hugging Face datasets.

    Uses sklearn.train_test_split for stratification, then wraps each part
    as a Dataset and applies tokenization.
    """
    if "sequence" not in df_all.columns or "label" not in df_all.columns:
        raise KeyError("df_all must contain 'sequence' and 'label' columns")

    # Stratified split on integer labels
    train_df, val_df = train_test_split(
        df_all,
        test_size=test_size,
        stratify=df_all["label"],
        random_state=42,
    )

    ds_train = Dataset.from_pandas(train_df[["sequence", "label"]], preserve_index=False)
    ds_val = Dataset.from_pandas(val_df[["sequence", "label"]], preserve_index=False)

    def tokenize_batch(batch):
        enc = tokenizer(
            batch["sequence"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        enc["labels"] = batch["label"]
        return enc

    train_tok = ds_train.map(tokenize_batch, batched=True, remove_columns=["sequence", "label"])
    val_tok = ds_val.map(tokenize_batch, batched=True, remove_columns=["sequence", "label"])

    return train_tok, val_tok


def compute_metrics(eval_pred):
    """
    Compute simple classification metrics for evaluation.
    """
    from sklearn.metrics import accuracy_score, f1_score

    preds_raw, labels = eval_pred
    # Handle tuple-of-arrays case
    if isinstance(preds_raw, (tuple, list)):
        logits = preds_raw[0]
    else:
        logits = preds_raw

    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_micro = f1_score(labels, preds, average="micro")

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
    }


def make_training_args(output_dir, num_epochs=3):
    """
    Small, stable TrainingArguments configuration for Colab.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_steps=10,
        report_to=["none"],
    )


def build_trainer_for_baseline(model_id, num_epochs=3):
    """
    Build Trainer for a simple baseline on gene_type tasks.
    """
    print(f"Building Trainer for {model_id}")

    df_all, label2id = build_baseline_dataframe()

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    train_ds, val_ds = build_hf_datasets(df_all, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=len(label2id),
    )

    args = make_training_args(
        BASELINE_OUT_ROOT / model_id.replace("/", "_"),
        num_epochs=num_epochs,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    return trainer, label2id, df_all


7. finetuning

In [ ]:
# Step 7: Run baseline finetuning and save metrics and model

BASELINE_MODEL_ID = "zhihan1996/DNABERT-2-117M"


def run_baseline_finetuning(model_id=BASELINE_MODEL_ID, num_epochs=3):
    trainer, label2id, df_all = build_trainer_for_baseline(model_id=model_id, num_epochs=num_epochs)

    print(f"Starting training on {len(df_all)} sequences with labels {label2id}")

    trainer.train()

    metrics = trainer.evaluate()
    print("Evaluation metrics:", metrics)

    metrics_path = BASELINE_OUT_ROOT / "baseline_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    model_dir = BASELINE_OUT_ROOT / "baseline_model"
    trainer.save_model(str(model_dir))

    print(f"Saved metrics to {metrics_path}")
    print(f"Saved finetuned model to {model_dir}")

    return str(metrics_path), str(model_dir), metrics


metrics_path, model_dir, metrics = run_baseline_finetuning(
    model_id=BASELINE_MODEL_ID,
    num_epochs=3,
)


Building Trainer for zhihan1996/DNABERT-2-117M
[SEQ] task=human_gene_type example sequence (first row, first 60 nt):
GTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTCACATCA
[LBL] gene type mapping for human_gene_type: {'cds': 0, 'rRNA': 1, 'tRNA': 2}
[SEQ] task=multispecies_gene_type example sequence (first row, first 60 nt):
TAACAAAGTAAGCTAATTTAAGCTAGTGGGTTCATACCCCAAAAATGAATTTTTCCTTTG
[LBL] gene type mapping for multispecies_gene_type: {'cds': 0, 'rRNA': 1, 'tRNA': 2}
[LBL] unified gene_type label2id: {0: 0, 1: 1, 2: 2}
Baseline dataframe size: 5111 rows


Map:   0%|          | 0/4088 [00:00<?, ? examples/s]

Map:   0%|          | 0/1023 [00:00<?, ? examples/s]

The repository zhihan1996/DNABERT-2-117M contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/zhihan1996/DNABERT-2-117M .
 You can inspect the repository content at https://hf.co/zhihan1996/DNABERT-2-117M.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.embeddings.position_embeddings.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.1.attention.self.key.bias', 'bert.encoder.layer.1.attention.self.key.weight', 'bert.encoder.layer.1.attention.self.query.bias', 'bert.encoder.layer.1.attention.self.query.weight', 'bert.encoder.layer.1.at

Starting training on 5111 sequences with labels {0: 0, 1: 1, 2: 2}


Step,Training Loss
10,1.067300
20,0.468900
30,0.236700
40,0.270500
50,0.281900
60,0.328000
70,0.237900
80,0.230200
90,0.289700
100,0.737900


Evaluation metrics: {'eval_loss': 0.043272413313388824, 'eval_accuracy': 0.9912023460410557, 'eval_f1_macro': 0.979614631333413, 'eval_f1_micro': 0.9912023460410557, 'eval_runtime': 7.28, 'eval_samples_per_second': 140.523, 'eval_steps_per_second': 8.791, 'epoch': 3.0}
Saved metrics to mitogpt_runs_drive/baseline_gene_type/baseline_metrics.json
Saved finetuned model to mitogpt_runs_drive/baseline_gene_type/baseline_model


In [ ]:
# Compress both result folders into a single zip
!zip -r mitogpt_results.zip mitogpt_runs_dnabert_baseline mitogpt_runs_drive

# Download the zip to local machine
from google.colab import files
files.download("mitogpt_results.zip")


  adding: mitogpt_runs_dnabert_baseline/ (stored 0%)
  adding: mitogpt_runs_drive/ (stored 0%)
  adding: mitogpt_runs_drive/baseline_gene_type/ (stored 0%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/ (stored 0%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/ (stored 0%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/rng_state.pth (deflated 26%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/configuration_bert.py (deflated 53%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/tokenizer_config.json (deflated 75%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/model.safetensors (deflated 7%)
  adding: mitogpt_runs_drive/baseline_gene_type/zhihan1996_DNABERT-2-117M/checkpoint-768/special_tokens_map.json (deflated 80%)
  adding: mitogpt_runs_drive/baselin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>